# LeetCode 1909 — Remove One Element to Make the Array Strictly Increasing bằng Python

Link đề: https://leetcode.com/problems/remove-one-element-to-make-the-array-strictly-increasing/

Độ khó: **Easy** · Chủ đề: Array

```python
class Solution:
    def canBeIncreasing(self, nums: list[int]) -> bool:

```

**Mục tiêu học:**

1. Hiểu **tăng nghiêm ngặt** (strictly increasing): `nums[i-1] < nums[i]` — dấu `<`, không phải `<=`.
2. Viết brute force "thử xóa từng phần tử" `O(n²)` và thấy rõ phần việc bị lặp.
3. Insight: chỉ cần nhìn **vị trí vi phạm đầu tiên** — phần tử bị xóa bắt buộc là một trong hai phần tử ở đó.
4. Lời giải tham lam một lượt `O(n)`: khi gặp "dốc xuống", quyết định xóa **phần tử trước** hay **phần tử hiện tại** dựa vào `nums[i-2]`.
5. Một góc nhìn sáng tạo: **dãy con tăng dài nhất (LIS)** phải có độ dài `>= n - 1`.

## 1. Bài toán yêu cầu gì?

Cho mảng `nums`. Trả `True` nếu **xóa đúng một phần tử** thì mảng trở thành tăng nghiêm ngặt; nếu mảng **đã** tăng nghiêm ngặt thì cũng trả `True` (xóa phần tử cuối vẫn giữ tính tăng).

| nums | Kết quả | Vì sao |
|---|:---:|---|
| `[1,2,10,5,7]` | `True` | xóa `10` → `[1,2,5,7]` |
| `[2,3,1,2]` | `False` | thử xóa từng phần tử, không lần nào tăng nghiêm ngặt |
| `[1,1,1]` | `False` | xóa bất kỳ còn `[1,1]` — bằng nhau không phải tăng nghiêm ngặt |
| `[1,2,3]` | `True` | đã tăng nghiêm ngặt |
| `[105,924,32,968]` | `True` | xóa `32` |

**Ràng buộc:** `2 <= len(nums) <= 1000`, `1 <= nums[i] <= 1000`.

In [1]:
examples = [
    ([1, 2, 10, 5, 7], True),
    ([2, 3, 1, 2], False),
    ([1, 1, 1], False),
    ([1, 2, 3], True),
    ([105, 924, 32, 968], True),
]
for nums, expected in examples:
    print(f"{nums} -> kỳ vọng {expected}")

[1, 2, 10, 5, 7] -> kỳ vọng True
[2, 3, 1, 2] -> kỳ vọng False
[1, 1, 1] -> kỳ vọng False
[1, 2, 3] -> kỳ vọng True
[105, 924, 32, 968] -> kỳ vọng True


## 2. Nền tảng Python cần biết

### 2.1. Kiểm tra tăng nghiêm ngặt

Cách rõ ràng nhất: duyệt `i` từ 1, kiểm tra `nums[i-1] < nums[i]`.

Cách gọn: `zip(nums, nums[1:])` tạo các **cặp kề nhau** `(nums[0], nums[1]), (nums[1], nums[2]), ...`, rồi `all(a < b ...)`.

In [2]:
nums = [1, 2, 10, 5, 7]
print(list(zip(nums, nums[1:])))

def is_strictly_increasing(arr):
    return all(a < b for a, b in zip(arr, arr[1:]))

print(is_strictly_increasing([1, 2, 5, 7]))   # True
print(is_strictly_increasing([1, 2, 2, 7]))   # False: 2 < 2 sai
print(is_strictly_increasing([5]))            # True: không có cặp nào

[(1, 2), (2, 10), (10, 5), (5, 7)]
True
False
True


Lưu ý `all([])` là `True`: mảng 0 hoặc 1 phần tử không có cặp nào vi phạm, nên coi là tăng nghiêm ngặt.

### 2.2. Tạo mảng đã xóa phần tử thứ `i`

`nums[:i] + nums[i+1:]` — ghép phần trước `i` và phần sau `i`. Tạo list **mới**, không đụng tới `nums`.

In [3]:
nums = [1, 2, 10, 5, 7]
for i in range(len(nums)):
    print(f"xóa i={i} ({nums[i]:2}): {nums[:i] + nums[i + 1:]}")
print("nums gốc:", nums)

xóa i=0 ( 1): [2, 10, 5, 7]
xóa i=1 ( 2): [1, 10, 5, 7]
xóa i=2 (10): [1, 2, 5, 7]
xóa i=3 ( 5): [1, 2, 10, 7]
xóa i=4 ( 7): [1, 2, 10, 5]
nums gốc: [1, 2, 10, 5, 7]


### 2.3. Cẩn thận: `list.pop(i)` / `del` **sửa** list gốc

Nếu dùng `nums.pop(i)` trong vòng lặp thử, list gốc bị thay đổi — lần thử sau sai. Muốn dùng `pop` phải sao chép trước: `tmp = nums[:]`.

In [4]:
a = [3, 1, 2]
tmp = a[:]
tmp.pop(0)
print("tmp:", tmp, "| a:", a)

tmp: [1, 2] | a: [3, 1, 2]


### 2.4. Từ "vi phạm" (dốc xuống)

Gọi cặp kề `(nums[i-1], nums[i])` là **vi phạm** nếu `nums[i-1] >= nums[i]`. Mảng tăng nghiêm ngặt ⇔ không có vi phạm.

In [5]:
def violations(arr):
    return [i for i in range(1, len(arr)) if arr[i - 1] >= arr[i]]

for nums, _ in examples:
    print(nums, "-> vi phạm tại i =", violations(nums))

[1, 2, 10, 5, 7] -> vi phạm tại i = [3]
[2, 3, 1, 2] -> vi phạm tại i = [2]
[1, 1, 1] -> vi phạm tại i = [1, 2]
[1, 2, 3] -> vi phạm tại i = []
[105, 924, 32, 968] -> vi phạm tại i = [2]


## 3. Tính bằng tay

**`[1,2,10,5,7]`:** vi phạm duy nhất tại `i = 3` (`10 >= 5`). Thử:

- xóa `nums[2] = 10` → `[1,2,5,7]` ✓.
- xóa `nums[3] = 5` → `[1,2,10,7]` ✗ (`10 >= 7`).

→ `True`.

**`[2,3,1,2]`:** vi phạm tại `i = 2` (`3 >= 1`). Thử:

- xóa `3` → `[2,1,2]` ✗.
- xóa `1` → `[2,3,2]` ✗.

Xóa phần tử nào khác (`nums[0]` hay `nums[3]`) thì cặp `3, 1` vẫn kề nhau → chắc chắn ✗. → `False`.

Câu hỏi gợi mở: tại sao chỉ cần thử 2 phần tử ở vị trí vi phạm **đầu tiên**?

## 4. Cách 1 — Brute force: thử xóa từng phần tử

Đúng như hint của đề: với mỗi `i`, tạo mảng đã xóa `nums[i]`, kiểm tra tăng nghiêm ngặt.

In [6]:
def can_be_increasing_brute(nums):
    for i in range(len(nums)):
        if is_strictly_increasing(nums[:i] + nums[i + 1:]):
            return True
    return False

for nums, expected in examples:
    print(nums, "->", can_be_increasing_brute(nums), "| kỳ vọng", expected)

[1, 2, 10, 5, 7] -> True | kỳ vọng True
[2, 3, 1, 2] -> False | kỳ vọng False
[1, 1, 1] -> False | kỳ vọng False
[1, 2, 3] -> True | kỳ vọng True
[105, 924, 32, 968] -> True | kỳ vọng True


### Code nộp LeetCode — Cách 1

In [7]:
class Solution:
    def canBeIncreasing(self, nums: list[int]) -> bool:
        for i in range(len(nums)):
            rest = nums[:i] + nums[i + 1:]
            if all(a < b for a, b in zip(rest, rest[1:])):
                return True
        return False


print(Solution().canBeIncreasing([1, 2, 10, 5, 7]))  # True

True


**Độ phức tạp:** `n` lần thử, mỗi lần tạo mảng mới `O(n)` và kiểm tra `O(n)` → `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n ≤ 1000` là ~10^6 phép — vẫn Accepted.

**Nút thắt:** đa số lần thử là vô ích. Nếu cặp vi phạm `(i-1, i)` còn nằm cạnh nhau sau khi xóa, chắc chắn thất bại. Ta đang thử `n` vị trí trong khi chỉ **2** vị trí có hy vọng.

## 5. Insight — chỉ cần xét vi phạm đầu tiên

Giả sử vi phạm đầu tiên ở cặp `(i-1, i)`, tức `nums[i-1] >= nums[i]`.

- Nếu xóa phần tử `k` với `k ≠ i-1` và `k ≠ i`, hai phần tử `nums[i-1]`, `nums[i]` vẫn **kề nhau** trong mảng mới → vẫn vi phạm.
- Vậy phần tử bị xóa **bắt buộc** là `nums[i-1]` hoặc `nums[i]`.

→ Chỉ cần thử 2 lần, mỗi lần `O(n)`.

In [8]:
nums = [2, 3, 1, 2]
first = violations(nums)[0]
print("vi phạm đầu tiên tại i =", first, f"(cặp {nums[first - 1]}, {nums[first]})")
for k in range(len(nums)):
    rest = nums[:k] + nums[k + 1:]
    tag = "ứng viên" if k in (first - 1, first) else "chắc chắn thất bại"
    print(f"  xóa k={k}: {rest} -> {is_strictly_increasing(rest)} ({tag})")

vi phạm đầu tiên tại i = 2 (cặp 3, 1)
  xóa k=0: [3, 1, 2] -> False (chắc chắn thất bại)
  xóa k=1: [2, 1, 2] -> False (ứng viên)
  xóa k=2: [2, 3, 2] -> False (ứng viên)
  xóa k=3: [2, 3, 1] -> False (chắc chắn thất bại)


## 6. Cách 2 — Tìm vi phạm đầu tiên, thử đúng 2 ứng viên

In [9]:
def can_be_increasing_first_violation(nums):
    for i in range(1, len(nums)):
        if nums[i - 1] >= nums[i]:
            without_prev = nums[:i - 1] + nums[i:]
            without_cur = nums[:i] + nums[i + 1:]
            return is_strictly_increasing(without_prev) or is_strictly_increasing(without_cur)
    return True          # không có vi phạm: đã tăng nghiêm ngặt

for nums, expected in examples:
    print(nums, "->", can_be_increasing_first_violation(nums), "| kỳ vọng", expected)

[1, 2, 10, 5, 7] -> True | kỳ vọng True
[2, 3, 1, 2] -> False | kỳ vọng False
[1, 1, 1] -> False | kỳ vọng False
[1, 2, 3] -> True | kỳ vọng True
[105, 924, 32, 968] -> True | kỳ vọng True


**Độ phức tạp Cách 2:** tìm vi phạm `O(n)` + hai lần kiểm tra `O(n)` → `O(n)` thời gian; `O(n)` bộ nhớ cho hai mảng tạm.

Có thể bỏ luôn mảng tạm không? Có — bằng cách **quyết định ngay** khi gặp vi phạm, nhìn thêm `nums[i-2]`.

## 7. Lời giải chính — Tham lam một lượt, `O(1)` bộ nhớ

Duyệt `i` từ 1, giữ:

- `prev` = phần tử **giữ lại** gần nhất (ban đầu `nums[0]`).
- `removed` = đã xóa phần tử nào chưa.

Khi `nums[i] > prev`: ổn, `prev = nums[i]`.

Khi `nums[i] <= prev` (vi phạm):

1. Nếu `removed` đã `True` → lần vi phạm thứ hai → `False`.
2. Đánh dấu `removed = True`, rồi chọn xóa ai:
   - **Xóa `nums[i-1]`** được nếu `i < 2` (không có ai phía trước) hoặc `nums[i-2] < nums[i]`. Khi đó `prev = nums[i]`.
   - Ngược lại **xóa `nums[i]`**: `prev` giữ nguyên.

Vì sao **ưu tiên xóa `nums[i-1]`**? Vì `nums[i] <= nums[i-1]`, giữ `nums[i]` làm `prev` cho giá trị **nhỏ hơn hoặc bằng** → phần còn lại dễ tăng hơn. Đây là lựa chọn tham lam "để lại cửa rộng nhất".

In [10]:
def can_be_increasing_greedy(nums):
    removed = False
    prev = nums[0]
    for i in range(1, len(nums)):
        if nums[i] > prev:
            prev = nums[i]
            continue
        if removed:
            return False
        removed = True
        if i < 2 or nums[i - 2] < nums[i]:
            prev = nums[i]          # xóa nums[i-1]
        # else: xóa nums[i], prev giữ nguyên
    return True

for nums, expected in examples:
    print(nums, "->", can_be_increasing_greedy(nums), "| kỳ vọng", expected)

[1, 2, 10, 5, 7] -> True | kỳ vọng True
[2, 3, 1, 2] -> False | kỳ vọng False
[1, 1, 1] -> False | kỳ vọng False
[1, 2, 3] -> True | kỳ vọng True
[105, 924, 32, 968] -> True | kỳ vọng True


### Vì sao lúc vi phạm đầu tiên, `prev` đúng bằng `nums[i-1]` và `nums[i-2]` chắc chắn được giữ?

Trước vi phạm đầu tiên, mọi phần tử đều được giữ và `prev` luôn được cập nhật bằng phần tử vừa duyệt → `prev = nums[i-1]`, còn `nums[i-2]` cũng là phần tử được giữ ngay trước đó. Sau khi đã xóa một lần, lần vi phạm tiếp theo trả `False` ngay, nên ta **không bao giờ** cần `nums[i-2]` ở trạng thái đã xóa.

### Cách 4 (sáng tạo) — Dãy con tăng dài nhất (LIS)

Xóa **đúng một** phần tử để còn tăng nghiêm ngặt ⇔ tồn tại dãy con tăng nghiêm ngặt dài `n - 1` ⇔ `LIS(nums) >= n - 1`.

Tính LIS bằng mảng `tails` + `bisect_left` (`O(n log n)`): `tails[k]` = phần tử cuối nhỏ nhất của một dãy con tăng dài `k+1`. `bisect_left(a, x)` trả vị trí đầu tiên trong list **đã sắp** `a` mà `a[pos] >= x`.

In [11]:
from bisect import bisect_left

print(bisect_left([1, 3, 5, 7], 5), bisect_left([1, 3, 5, 7], 6), bisect_left([1, 3, 5, 7], 9))

def lis_length(nums):
    tails = []
    for x in nums:
        pos = bisect_left(tails, x)     # bisect_left -> tăng NGHIÊM NGẶT
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
    return len(tails)

def can_be_increasing_lis(nums):
    return lis_length(nums) >= len(nums) - 1

for nums, expected in examples:
    print(nums, "LIS =", lis_length(nums), "->", can_be_increasing_lis(nums), "| kỳ vọng", expected)

2 3 4
[1, 2, 10, 5, 7] LIS = 4 -> True | kỳ vọng True
[2, 3, 1, 2] LIS = 2 -> False | kỳ vọng False
[1, 1, 1] LIS = 1 -> False | kỳ vọng False
[1, 2, 3] LIS = 3 -> True | kỳ vọng True
[105, 924, 32, 968] LIS = 3 -> True | kỳ vọng True


LIS là "dao mổ trâu" cho bài Easy, nhưng nó khái quát ngay sang biến thể "xóa **tối đa k** phần tử": điều kiện là `LIS >= n - k`.

## 8. TRACE từng bước (tham lam)

Ví dụ `[1,2,10,5,7]`:

| i | nums[i] | prev trước | so sánh | removed | quyết định | prev sau |
|---:|---:|---:|---|:---:|---|---:|
| 1 | 2 | 1 | 2 > 1 | F | giữ | 2 |
| 2 | 10 | 2 | 10 > 2 | F | giữ | 10 |
| 3 | 5 | 10 | 5 <= 10 vi phạm | → T | `nums[1]=2 < 5` → xóa `10` | 5 |
| 4 | 7 | 5 | 7 > 5 | T | giữ | 7 |

→ `True`.

In [12]:
def trace_greedy(nums):
    removed = False
    prev = nums[0]
    print(f"nums = {nums}, prev ban đầu = {prev}")
    for i in range(1, len(nums)):
        x = nums[i]
        if x > prev:
            print(f"  i={i}: {x} > prev={prev} -> giữ, prev={x}")
            prev = x
            continue
        if removed:
            print(f"  i={i}: {x} <= prev={prev}, đã xóa rồi -> False")
            return False
        removed = True
        if i < 2 or nums[i - 2] < x:
            why = "i < 2" if i < 2 else f"nums[i-2]={nums[i - 2]} < {x}"
            print(f"  i={i}: {x} <= prev={prev} -> vi phạm; {why} -> xóa nums[{i - 1}]={nums[i - 1]}, prev={x}")
            prev = x
        else:
            print(f"  i={i}: {x} <= prev={prev} -> vi phạm; nums[i-2]={nums[i - 2]} >= {x} -> xóa nums[{i}]={x}, prev giữ {prev}")
    print("  Hết mảng -> True")
    return True

trace_greedy([1, 2, 10, 5, 7])

nums = [1, 2, 10, 5, 7], prev ban đầu = 1
  i=1: 2 > prev=1 -> giữ, prev=2
  i=2: 10 > prev=2 -> giữ, prev=10
  i=3: 5 <= prev=10 -> vi phạm; nums[i-2]=2 < 5 -> xóa nums[2]=10, prev=5
  i=4: 7 > prev=5 -> giữ, prev=7
  Hết mảng -> True


True

### Trace nhánh "xóa phần tử hiện tại" và nhánh thất bại

In [13]:
trace_greedy([105, 924, 32, 968])
print()
trace_greedy([2, 3, 1, 2])
print()
trace_greedy([1, 1, 1])

nums = [105, 924, 32, 968], prev ban đầu = 105
  i=1: 924 > prev=105 -> giữ, prev=924
  i=2: 32 <= prev=924 -> vi phạm; nums[i-2]=105 >= 32 -> xóa nums[2]=32, prev giữ 924
  i=3: 968 > prev=924 -> giữ, prev=968
  Hết mảng -> True

nums = [2, 3, 1, 2], prev ban đầu = 2
  i=1: 3 > prev=2 -> giữ, prev=3
  i=2: 1 <= prev=3 -> vi phạm; nums[i-2]=2 >= 1 -> xóa nums[2]=1, prev giữ 3
  i=3: 2 <= prev=3, đã xóa rồi -> False

nums = [1, 1, 1], prev ban đầu = 1
  i=1: 1 <= prev=1 -> vi phạm; i < 2 -> xóa nums[0]=1, prev=1
  i=2: 1 <= prev=1, đã xóa rồi -> False


False

Ở `[105,924,32,968]`: `nums[0] = 105 >= 32` nên không thể xóa `924` (vì `105, 32` sẽ kề nhau và giảm) → xóa `32`, `prev` giữ `924`, sau đó `968 > 924` ✓.

## 9. Code nộp LeetCode — lời giải chính (tham lam)

In [14]:
class Solution:
    def canBeIncreasing(self, nums: list[int]) -> bool:
        removed = False
        prev = nums[0]
        for i in range(1, len(nums)):
            if nums[i] > prev:
                prev = nums[i]
                continue
            if removed:
                return False
            removed = True
            if i < 2 or nums[i - 2] < nums[i]:
                prev = nums[i]
        return True


sol = Solution()
for nums, expected in examples:
    assert sol.canBeIncreasing(nums) == expected
print("Solution (tham lam) PASS")

Solution (tham lam) PASS


## 10. Vì sao đúng?

**Bổ đề (mục 5).** Nếu vi phạm đầu tiên ở cặp `(i-1, i)` thì phần tử bị xóa phải là `nums[i-1]` hoặc `nums[i]`. Hệ quả: có **hai** vi phạm "không thể gỡ chung" thì trả `False`.

**Quyết định tham lam đúng.** Tại vi phạm đầu tiên, phần `nums[0..i-1]` tăng nghiêm ngặt.

- Xóa `nums[i-1]` hợp lệ cục bộ khi `nums[i-2] < nums[i]` (hoặc không có `nums[i-2]`). Kết quả: phần đã duyệt kết thúc bằng `nums[i]`.
- Xóa `nums[i]` luôn hợp lệ cục bộ. Kết quả: phần đã duyệt kết thúc bằng `nums[i-1]`.

Vì `nums[i] <= nums[i-1]`, kết thúc bằng `nums[i]` **không tệ hơn**: bất kỳ phần đuôi nào nối được sau `nums[i-1]` cũng nối được sau `nums[i]` (vì phần tử kế tiếp `> nums[i-1] >= nums[i]`). Nên khi cả hai đều hợp lệ, chọn xóa `nums[i-1]` không làm mất lời giải nào.

**Sau khi đã xóa,** phần còn lại phải tăng nghiêm ngặt so với `prev` mà không xóa thêm — đúng như vòng lặp kiểm tra. Nếu không vi phạm nào xuất hiện, mảng đã tăng → `True` (xóa phần tử cuối vẫn tăng).

**Cách 4 (LIS):** xóa một phần tử để được dãy tăng nghiêm ngặt ⇔ có một dãy con tăng nghiêm ngặt độ dài `n-1` ⇔ `LIS ≥ n-1` (LIS = `n` cũng thỏa, vì xóa phần tử cuối).

## 11. Độ phức tạp và bảng so sánh

`n = len(nums)` (≤ 1000).

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|:---:|:---:|---|
| 1. Brute force | thử xóa từng `i` | `O(n²)` | `O(n)` | đúng hint, dễ viết |
| 2. Vi phạm đầu tiên | thử 2 ứng viên | `O(n)` | `O(n)` | bổ đề đẹp, dễ chứng minh |
| 3. Tham lam một lượt | quyết định qua `nums[i-2]` | `O(n)` | `O(1)` | tối ưu, hay hỏi phỏng vấn |
| 4. LIS + bisect | `LIS >= n-1` | `O(n log n)` | `O(n)` | khái quát cho "xóa tối đa k" |

## 12. Bộ test kiểm tra

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:34} PASS")

solvers = [can_be_increasing_brute, can_be_increasing_first_violation,
           can_be_increasing_greedy, can_be_increasing_lis]

tests = [
    ("ví dụ 1", [1, 2, 10, 5, 7], True),
    ("ví dụ 2", [2, 3, 1, 2], False),
    ("ví dụ 3", [1, 1, 1], False),
    ("đã tăng", [1, 2, 3], True),
    ("phải xóa phần tử hiện tại", [105, 924, 32, 968], True),
    ("n=2 giảm", [2, 1], True),
    ("n=2 bằng", [5, 5], True),
    ("vi phạm ở đầu", [10, 1, 2, 3], True),
    ("vi phạm ở cuối", [1, 2, 3, 0], True),
    ("hai vi phạm xa nhau", [1, 3, 2, 4, 6, 5], False),
    ("bằng nhau ở giữa", [1, 2, 2, 3], True),
    ("xóa phần tử đỉnh", [1, 2, 5, 3, 4], True),
    ("phải chọn đúng nhánh", [3, 4, 1, 2], False),
]
for name, nums, e in tests:
    for f in solvers:
        assert f(nums) == e, (name, f.__name__)
    check(name + " (4 cách)", can_be_increasing_greedy(nums), e)

ví dụ 1 (4 cách)                   PASS
ví dụ 2 (4 cách)                   PASS
ví dụ 3 (4 cách)                   PASS
đã tăng (4 cách)                   PASS
phải xóa phần tử hiện tại (4 cách) PASS
n=2 giảm (4 cách)                  PASS
n=2 bằng (4 cách)                  PASS
vi phạm ở đầu (4 cách)             PASS
vi phạm ở cuối (4 cách)            PASS
hai vi phạm xa nhau (4 cách)       PASS
bằng nhau ở giữa (4 cách)          PASS
xóa phần tử đỉnh (4 cách)          PASS
phải chọn đúng nhánh (4 cách)      PASS


In [16]:
import random

random.seed(1909)
mismatch = 0
for _ in range(4000):
    n_ = random.randint(2, 8)
    nums = [random.randint(1, 6) for _ in range(n_)]
    expected = can_be_increasing_brute(nums)
    if any(f(nums) != expected for f in solvers[1:]):
        mismatch += 1
print("Số ca lệch so với brute force:", mismatch)
assert mismatch == 0

Số ca lệch so với brute force: 0


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<` (tăng **không** nghiêm ngặt)

```python
all(a <= b for a, b in zip(arr, arr[1:]))   # SAI: chấp nhận [1, 1]
```

In [17]:
def wrong_non_strict(nums):
    for i in range(len(nums)):
        rest = nums[:i] + nums[i + 1:]
        if all(a <= b for a, b in zip(rest, rest[1:])):
            return True
    return False

print("Sai :", wrong_non_strict([1, 1, 1]), "| Đúng:", can_be_increasing_brute([1, 1, 1]))

Sai : True | Đúng: False


### Lỗi 2: Chỉ đếm số vi phạm `<= 1`

```python
return sum(nums[i-1] >= nums[i] for i in range(1, n)) <= 1   # SAI
```

Một vi phạm duy nhất **chưa chắc** gỡ được: `[3,4,1,2]` chỉ có một vi phạm (`4 >= 1`) nhưng xóa `4` → `[3,1,2]`, xóa `1` → `[3,4,2]`, đều hỏng.

In [18]:
def wrong_count_only(nums):
    return sum(nums[i - 1] >= nums[i] for i in range(1, len(nums))) <= 1

print("Sai :", wrong_count_only([3, 4, 1, 2]), "| Đúng:", can_be_increasing_brute([3, 4, 1, 2]))

Sai : True | Đúng: False


### Lỗi 3: Luôn xóa phần tử hiện tại (hoặc luôn xóa phần tử trước)

Mỗi chiến lược "cứng" đều có phản ví dụ:

- Luôn xóa `nums[i]`: `[1,2,10,5,7]` → giữ `10`, rồi `5`, `7` đều `<= 10` → trả `False` sai.
- Luôn xóa `nums[i-1]` (không xét `nums[i-2]`): `[3,4,1,2]` → bỏ `4`, đặt `prev = 1`, rồi `2 > 1` → trả `True`. Nhưng mảng thật sau khi xóa là `[3,1,2]` — `3 >= 1` vẫn vi phạm, đáp án đúng là `False`. Code quên rằng `nums[i-2] = 3` giờ đứng cạnh `nums[i] = 1`.

In [19]:
def wrong_always_remove_cur(nums):
    removed, prev = False, nums[0]
    for i in range(1, len(nums)):
        if nums[i] > prev:
            prev = nums[i]
        elif removed:
            return False
        else:
            removed = True           # luôn bỏ nums[i]
    return True

def wrong_always_remove_prev(nums):
    removed, prev = False, nums[0]
    for i in range(1, len(nums)):
        if nums[i] > prev:
            prev = nums[i]
        elif removed:
            return False
        else:
            removed = True
            prev = nums[i]           # luôn bỏ nums[i-1], quên kiểm tra nums[i-2]
    return True

print("Luôn xóa hiện tại, [1,2,10,5,7]:", wrong_always_remove_cur([1, 2, 10, 5, 7]), "| đúng: True")
print("Luôn xóa phần tử trước, [3,4,1,2]:", wrong_always_remove_prev([3, 4, 1, 2]), "| đúng: False")

Luôn xóa hiện tại, [1,2,10,5,7]: False | đúng: True
Luôn xóa phần tử trước, [3,4,1,2]: True | đúng: False


### Lỗi 4: `pop` trên list gốc khi thử brute force

```python
for i in range(len(nums)):
    nums.pop(i)            # SAI: nums bị co lại sau mỗi lần thử
```

Sửa: tạo mảng mới `nums[:i] + nums[i+1:]` hoặc sao chép `tmp = nums[:]` trước khi `pop`.

### Lỗi 5: Quên trường hợp `i = 1` khi đọc `nums[i-2]`

Với `i = 1`, `nums[i-2]` là `nums[-1]` — **phần tử cuối mảng** trong Python (không báo lỗi!), dẫn tới quyết định sai một cách âm thầm. Luôn kiểm tra `i < 2` trước.

In [20]:
def wrong_no_i_check(nums):
    removed, prev = False, nums[0]
    for i in range(1, len(nums)):
        if nums[i] > prev:
            prev = nums[i]; continue
        if removed:
            return False
        removed = True
        if nums[i - 2] < nums[i]:      # SAI khi i = 1: nums[-1]
            prev = nums[i]
    return True

x = [5, 1, 2, 3, 9]
print("nums[-1] khi i=1 là", x[-1])
print("Sai :", wrong_no_i_check(x), "| Đúng:", can_be_increasing_greedy(x))

nums[-1] khi i=1 là 9
Sai : False | Đúng: True


## 14. Mẹo nhận dạng pattern và biến thể

- "Được sửa/xóa **tối đa một** phần tử để mảng thỏa tính chất cục bộ (tăng, không giảm, palindrome)" → tìm **vi phạm đầu tiên**, chỉ thử **hai** lựa chọn ở đó.
- Khi chọn giữa hai lựa chọn, nghĩ **tham lam**: để lại trạng thái "dễ tiếp tục" nhất (giá trị `prev` nhỏ nhất).
- "Xóa tối đa k phần tử" → LIS (`n - LIS <= k`).

Bài liên quan: 665 (Non-decreasing Array — **sửa** một phần tử, không giảm), 680 (Valid Palindrome II — xóa một ký tự), 300 (LIS), 1671.

### Biến thể: LeetCode 665 — sửa tối đa một phần tử để **không giảm**

Cùng khung tham lam, nhưng là `>` thay vì `>=` và "sửa" thay vì "xóa".

In [21]:
def check_possibility(nums):
    nums = nums[:]
    changed = False
    for i in range(1, len(nums)):
        if nums[i - 1] > nums[i]:
            if changed:
                return False
            changed = True
            if i < 2 or nums[i - 2] <= nums[i]:
                nums[i - 1] = nums[i]      # hạ phần tử trước
            else:
                nums[i] = nums[i - 1]      # nâng phần tử hiện tại
    return True

print(check_possibility([4, 2, 3]), check_possibility([4, 2, 1]), check_possibility([3, 4, 2, 3]))

True False False


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Dự đoán output

Không chạy code, `canBeIncreasing` trả gì với: `[1,3,2]`, `[3,1,2,1]`, `[1,2,3,4,1]`, `[2,2,2,3]`, `[100,21,100]`? Ghi vào `ex1_answers`.

In [22]:
ex1_answers = None   # TODO: list 5 bool

### Bài 2 — Điền chỗ trống: tìm vi phạm đầu tiên rồi thử 2 ứng viên

In [23]:
def ex2_can_be_increasing(nums):
    for i in range(1, len(nums)):
        if nums[i - 1] >= nums[i]:
            # TODO: tạo hai mảng ứng viên và kiểm tra
            pass
    return True

# for nums, e in examples: assert ex2_can_be_increasing(nums) == e

### Bài 3 — Trả về **chỉ số** cần xóa

Viết `index_to_remove(nums)`: trả về chỉ số **nhỏ nhất** mà xóa nó thì mảng tăng nghiêm ngặt, hoặc `-1`. (Mảng đã tăng thì trả `0`, vì xóa phần tử đầu vẫn tăng.)

In [24]:
def index_to_remove(nums):
    # TODO
    pass

# assert index_to_remove([1, 2, 10, 5, 7]) == 2
# assert index_to_remove([2, 3, 1, 2]) == -1
# assert index_to_remove([1, 2, 3]) == 0

### Bài 4 — Tự viết lời giải tham lam `O(1)` bộ nhớ (không nhìn code trên)

In [25]:
def ex4_can_be_increasing(nums):
    # TODO
    pass

# for nums, e in examples: assert ex4_can_be_increasing(nums) == e

### Bài 5 — Biến thể: xóa tối đa một phần tử để **không giảm** (`<=`)

In [26]:
def can_be_non_decreasing(nums):
    # TODO: giống tham lam nhưng vi phạm là nums[i] < prev
    pass

# assert can_be_non_decreasing([1, 1, 1]) == True
# assert can_be_non_decreasing([3, 4, 1, 2]) == False
# assert can_be_non_decreasing([1, 5, 2, 2]) == True

### Bài 6 — Mini challenge: xóa tối đa `k` phần tử

Viết `can_remove_k(nums, k)` trả `True` nếu xóa tối đa `k` phần tử thì mảng tăng nghiêm ngặt. Yêu cầu `O(n log n)`.

In [27]:
def can_remove_k(nums, k):
    # TODO: dùng LIS
    pass

# assert can_remove_k([5, 1, 2, 6, 3, 4], 2) == True
# assert can_remove_k([5, 4, 3, 2, 1], 3) == False

## 16. Gợi ý 3 mức

**Bài 1.** (1) Tìm vi phạm đầu tiên, thử xóa 2 phần tử. (2) `[3,1,2,1]` có vi phạm ở `(3,1)` và `(2,1)`. (3) `[100,21,100]`: xóa `21` → `[100,100]` không tăng nghiêm ngặt, nhưng xóa `100` đầu → `[21,100]` thì được — luôn thử **cả hai** ứng viên!

**Bài 2.** (1) Hai ứng viên: bỏ `i-1` hoặc bỏ `i`. (2) `nums[:i-1] + nums[i:]` và `nums[:i] + nums[i+1:]`. (3) `return inc(a) or inc(b)`.

**Bài 3.** (1) Brute force đủ nhanh. (2) Duyệt `i` tăng dần, trả `i` đầu tiên thỏa. (3) Không có → `-1`.

**Bài 4.** (1) Biến `prev`, `removed`. (2) Vi phạm lần hai → `False`. (3) Xóa `i-1` khi `i < 2 or nums[i-2] < nums[i]`.

**Bài 5.** (1) Đổi điều kiện vi phạm thành `nums[i] < prev`. (2) Điều kiện xóa `i-1`: `nums[i-2] <= nums[i]`. (3) Còn lại giống hệt.

**Bài 6.** (1) Dãy còn lại là một dãy con tăng. (2) Giữ lại nhiều nhất `LIS` phần tử. (3) `len(nums) - lis_length(nums) <= k`.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [28]:
ans_ex1 = [True, False, True, False, True]


def ans_ex2_can_be_increasing(nums):
    for i in range(1, len(nums)):
        if nums[i - 1] >= nums[i]:
            a = nums[:i - 1] + nums[i:]
            b_ = nums[:i] + nums[i + 1:]
            return is_strictly_increasing(a) or is_strictly_increasing(b_)
    return True


def ans_index_to_remove(nums):
    for i in range(len(nums)):
        if is_strictly_increasing(nums[:i] + nums[i + 1:]):
            return i
    return -1


def ans_ex4_can_be_increasing(nums):
    removed, prev = False, nums[0]
    for i in range(1, len(nums)):
        if nums[i] > prev:
            prev = nums[i]
            continue
        if removed:
            return False
        removed = True
        if i < 2 or nums[i - 2] < nums[i]:
            prev = nums[i]
    return True


def ans_can_be_non_decreasing(nums):
    removed, prev = False, nums[0]
    for i in range(1, len(nums)):
        if nums[i] >= prev:
            prev = nums[i]
            continue
        if removed:
            return False
        removed = True
        if i < 2 or nums[i - 2] <= nums[i]:
            prev = nums[i]
    return True


def ans_can_remove_k(nums, k):
    return len(nums) - lis_length(nums) <= k


assert ans_ex1 == [can_be_increasing_brute(x) for x in [[1, 3, 2], [3, 1, 2, 1], [1, 2, 3, 4, 1], [2, 2, 2, 3], [100, 21, 100]]]
for nums, e in examples:
    assert ans_ex2_can_be_increasing(nums) == e and ans_ex4_can_be_increasing(nums) == e
assert ans_index_to_remove([1, 2, 10, 5, 7]) == 2
assert ans_index_to_remove([2, 3, 1, 2]) == -1
assert ans_index_to_remove([1, 2, 3]) == 0
assert ans_can_be_non_decreasing([1, 1, 1]) and not ans_can_be_non_decreasing([3, 4, 1, 2])
assert ans_can_be_non_decreasing([1, 5, 2, 2])
assert ans_can_remove_k([5, 1, 2, 6, 3, 4], 2) and not ans_can_remove_k([5, 4, 3, 2, 1], 3)

# đối chiếu bài 5 với brute force
def brute_non_dec(nums):
    for i in range(len(nums)):
        r = nums[:i] + nums[i + 1:]
        if all(a <= b_ for a, b_ in zip(r, r[1:])):
            return True
    return False

random.seed(5)
for _ in range(2000):
    arr = [random.randint(1, 5) for _ in range(random.randint(2, 7))]
    assert ans_can_be_non_decreasing(arr) == brute_non_dec(arr), arr
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def canBeIncreasing(self, nums):
        removed, prev = False, nums[0]
        for i in range(1, len(nums)):
            if nums[i] > prev:              # ổn
                prev = nums[i]
                continue
            if removed:                     # vi phạm lần 2
                return False
            removed = True
            if i < 2 or nums[i - 2] < nums[i]:
                prev = nums[i]              # xóa nums[i-1] (ưu tiên)
            # else: xóa nums[i], prev giữ nguyên
        return True
```

**Khung tư duy:** "tối đa 1 lần sửa/xóa" → vi phạm đầu tiên → 2 lựa chọn → tham lam giữ `prev` nhỏ nhất.

**Checklist trước khi submit:**

- [ ] Dùng `<` (nghiêm ngặt), vi phạm là `nums[i] <= prev`?
- [ ] Kiểm tra `i < 2` trước khi đọc `nums[i-2]`?
- [ ] Không chỉ đếm số vi phạm?
- [ ] Không `pop` trên list gốc?
- [ ] Mảng đã tăng sẵn trả `True`? Tên method `canBeIncreasing`?